# Extracting, Transforming, and Loading CUAD Data into Dataframes - Erin Sur (09/23/2026)

Note: The team uses CUAD's official split (train_separate_questions.json / test.json) for all later steps. This notebook's split is kept for reference only.

#### Extracting Contracts to Split the Data into Training and Test 

In [1]:
import json
from sklearn.model_selection import train_test_split
import pandas as pd

I will first load the json file and start extracting the titles of each of the contracts.

In [2]:
with open('../data/CUADv1.json', 'r') as file:
    cuad = json.load(file)

unique_titles = []
for part in cuad['data']:
    unique_titles.append(part['title'])
unique_titles


['LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT',
 'WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION AND DISTRIBUTION AGREEMENT',
 'LohaCompanyltd_20191209_F-1_EX-10.16_11917878_EX-10.16_Supply Agreement',
 'CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WEB SITE HOSTING AGREEMENT',
 'NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT',
 'ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT AGREEMENT',
 'KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSULTING AGREEMENT',
 'VEONEER,INC_02_21_2020-EX-10.11-JOINT VENTURE AGREEMENT',
 'DovaPharmaceuticalsInc_20181108_10-Q_EX-10.2_11414857_EX-10.2_Promotion Agreement',
 'PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC LICENSING, DISTRIBUTION AND MARKETING AGREEMENT ',
 'MetLife, Inc. - Remarketing Agreement',
 'FTENETWORKS,INC_02_18_2016-EX-99.4-STRATEGIC ALLIANCE AGREEMENT',
 'DOMINIADVISORTRUST_02_18_2005-EX-99.(H)(2)-SPONSORSHIP AGREEMENT',
 'PfHospitalityGroupInc_20150923_10-12G_EX-10.1_9266710_EX-10.1_Franchise Agreement3',
 'CerenceInc_20191002_8-K_EX-

I will then split this list into train and test (80/20)

In [3]:
train_titles, test_titles = train_test_split(
    unique_titles,
    test_size=0.2,
    random_state=42,
)

In [4]:
train_titles

['InnerscopeHearingTechnologiesInc_20181109_8-K_EX-10.6_11419704_EX-10.6_Distributor Agreement',
 'OMINTO,INC_03_29_2004-EX-10-RESELLER AGREEMENT',
 'PRECIGEN,INC_01_22_2020-EX-99.1-JOINT FILING AGREEMENT',
 'ArcGroupInc_20171211_8-K_EX-10.1_10976103_EX-10.1_Sponsorship Agreement',
 'NETZEEINC_11_14_2002-EX-10.3-MAINTENANCE AGREEMENT',
 'IMMUNOMEDICSINC_08_07_2019-EX-10.1-PROMOTION AGREEMENT',
 'BellringBrandsInc_20190920_S-1_EX-10.12_11817081_EX-10.12_Manufacturing Agreement1',
 'ATHENSBANCSHARESCORP_11_02_2009-EX-1.2-AGENCY AGREEMENT , 2009',
 'AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1',
 'ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT',
 'ScansourceInc_20190822_10-K_EX-10.39_11793959_EX-10.39_Distributor Agreement',
 'ThriventVariableInsuranceAccountB_20190701_N-6_EX-99.D(IV)_11720968_EX-99.D(IV)_Endorsement Agreement',
 'ALLISONTRANSMISSIONHOLDINGSINC_12_15_2014-EX-99.1-COOPERATION AGREEMENT',
 'SPORTHALEYINC_09_29_1997-EX-10.2-10-ENDORSEMENT AGREEMENT',

In [5]:
test_titles

['VnueInc_20150914_8-K_EX-10.1_9259571_EX-10.1_Promotion Agreement',
 '2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement',
 'ZONDWINDSYSTEMPARTNERSLTDSERIES85-B_04_03_2006-EX-10-MANAGEMENT AND MAINTENANCE AGREEMENT',
 'ICORECONNECTINC_10_13_2010-EX-7.1-Strategic Alliance Agreement',
 'LejuHoldingsLtd_20140121_DRS (on F-1)_EX-10.26_8473102_EX-10.26_Content License Agreement1',
 'WPPPLC_04_30_2020-EX-4.28-SERVICE AGREEMENT',
 'PaperexchangeComInc_20000322_S-1A_EX-10.4_5202103_EX-10.4_Co-Branding Agreement',
 'ELECTRAMECCANICA VEHICLES CORP. - Manufacturing Agreement',
 'BLUEROCKRESIDENTIALGROWTHREIT,INC_06_01_2016-EX-1.1-AGENCY AGREEMENT',
 'ADMA BioManufacturing, LLC -  Amendment #3 to Manufacturing Agreement ',
 'ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-EX-10.13-JOINT VENTURE AGREEMENT',
 'ArcaUsTreasuryFund_20200207_N-2_EX-99.K5_11971930_EX-99.K5_Development Agreement',
 'RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT

In [6]:
#turning them into sets for faster checking
train_titles_set = set(train_titles)
test_titles_set = set(test_titles)
print("Number of training contracts: " + str(len(train_titles_set)))
print("Number of testing contracts: " + str(len(test_titles_set)))

Number of training contracts: 408
Number of testing contracts: 102


Now that I have two lists of contract title names, I will go back to the JSON and for every question and answer pair, route it according to whether its contract name is in train or test. 

In [7]:
train_data = []
test_data = []
for contract in cuad['data']:
    for paragraphs in contract['paragraphs']:
        for qa in paragraphs['qas']:
            # this layer has { answers, id, question, and is impossible } 
            # I will save it with the title and context
            temp_json = { 'title': contract['title'], 'context': paragraphs['context'], **qa}
            if(contract['title'] in train_titles_set):
                train_data.append(temp_json)
            else:
                test_data.append(temp_json)
train_data[0] # checking that output has title, context, question, answers, id, and is_impossible

{'title': 'WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION AND DISTRIBUTION AGREEMENT',
 'context': 'Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BEEN REQUESTED AS TO CERTAIN PORTIONS OF THIS DOCUMENT. EACH SUCH PORTION,  WHICH HAS BEEN OMITTED HEREIN AND REPLACED WITH AN ASTERISK [*], HAS BEEN FILED SEPARATELY WITH THE  SECURITIES AND EXCHANGE COMMISSION.     PROMOTION AND DISTRIBUTION AGREEMENT     This Promotion and Distribution Agreement including all exhibits (collectively referred to as the "Agreement"), effective as of 1 August 2011 (the  "Effective Date"), is made by and between Whitesmoke Inc., with registered offices/principle place of business at 501 Silverside Road, Suite 105,  Wilmington DE 19809, USA, ("Distributor"), and Google Inc whose principle place of business is at 1600 Amphitheatre Parkway, Mountain View,  CA 94043, USA ("Google").\n\n\n\n   "Bundle" means the Distribution Products bundled with the Distributor App(s).     "Chrome Browser" means the machine-readable bi

Now it is able to be flattened into pandas dataframes!

In [8]:
train_df = pd.DataFrame(train_data)
train_df.head(5)
train_df['context'].iloc[0]

'Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BEEN REQUESTED AS TO CERTAIN PORTIONS OF THIS DOCUMENT. EACH SUCH PORTION,  WHICH HAS BEEN OMITTED HEREIN AND REPLACED WITH AN ASTERISK [*], HAS BEEN FILED SEPARATELY WITH THE  SECURITIES AND EXCHANGE COMMISSION.     PROMOTION AND DISTRIBUTION AGREEMENT     This Promotion and Distribution Agreement including all exhibits (collectively referred to as the "Agreement"), effective as of 1 August 2011 (the  "Effective Date"), is made by and between Whitesmoke Inc., with registered offices/principle place of business at 501 Silverside Road, Suite 105,  Wilmington DE 19809, USA, ("Distributor"), and Google Inc whose principle place of business is at 1600 Amphitheatre Parkway, Mountain View,  CA 94043, USA ("Google").\n\n\n\n   "Bundle" means the Distribution Products bundled with the Distributor App(s).     "Chrome Browser" means the machine-readable binary code version of the Google Chrome browser provided to Distributor in connection with  this A

Now you all can start the inspection, cleaning, and tokenizing process. 